In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import h5py
from scipy.fft import fftn, ifftn, fftfreq
import illustris_python as il
from matplotlib.colors import LogNorm
import matplotlib.animation as animation
import matplotlib.patches as mpatches
import os
import matplotlib.ticker as ticker

In [ ]:
z_list = [0.0,0.1,0.2,0.3,0.4,0.5,0.7,1.0,1.5,2.0,3.01,4.01,5.0,6.01,7.01,8.01,9.0,10.0]
z_smol_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 1.5, 2.0, 3.01, 4.01, 5.0, 6.01, 8.01, 10.0]
snap_list = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,11,8,6,4]
snap_smol_list = [99,91,84,78,72,67,59,50,40,33,25,21,17,13,8,4]
smol_id = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,15,17]

colors = plt.colormaps['viridis'](np.linspace(0, 1, 4))
colors[3] = [1.0, 0.65, 0.0, 1.0]

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    'font.size': 12,
})

basePath = "/net/virgo01/data/users/folkertsma/MScThesis/data/PracticeData/TNG50-4-Dark/output"

L = 35.0 # cMpc/h
res = 128

dl  = L / res
print (f"The voxel width is {dl} cMpc/h")

norm_dens = []
voids = []
filaments = []
nodes = []
walls = []
dens = []

with h5py.File('/Users/users/roana/roana/BSc_Thesis/nexus_outputs.h5', 'r') as f:
    for red in z_list:
        norm_dens.append(np.transpose(f[f'z_{red}']['normalized_density'][:], (1,2,0)))
        dens.append(np.transpose(f[f'z_{red}']['density'][:], (1,2,0)))
        voids.append(np.transpose(f[f'z_{red}']['voids'][:], (1,2,0)))
        filaments.append(np.transpose(f[f'z_{red}']['filaments'][:], (1,2,0)))
        nodes.append(np.transpose(f[f'z_{red}']['nodes'][:], (1,2,0)))
        walls.append(np.transpose(f[f'z_{red}']['walls'][:], (1,2,0)))
        print(f"Suessfully Loaded Stuff for z={red}")

n = len(norm_dens)
print(f"{n} snapshots")

In [ ]:
id = 0

coords_z0 = il.snapshot.loadSubset(basePath, snap_list[id], 'dm', fields = ['Coordinates'])/1000 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# --- 1. DRAW PLOTS, LABELS, AND COLORBARS ONCE ---
d_vmin = np.min(np.log10(norm_dens))
d_vmax = np.max(np.log10(norm_dens))
im1 = ax1.imshow(np.log10(norm_dens[id][:,:,0]), origin="lower", cmap="inferno", extent=[0,35,0,35], vmin = d_vmin, vmax = d_vmax)
fig.colorbar(im1, ax=ax1, label=r"$\log (1+\delta)$")

counts, xedges, yedges = np.histogram2d([], [], bins=256, range=[[0,35], [0,35]])
im2 = ax2.imshow(counts.T, origin="lower", cmap="inferno", norm=LogNorm(vmin=1, vmax=10000), extent=[0,35,0,35])
fig.colorbar(im2, ax=ax2, label="Particle Counts")



plt.tight_layout()

# --- 2. JUST LOOP THROUGH THE 128 SLICES ---
def update(frame):
    # Swap out density slice
    ax1.clear()
    ax2.clear()
    ax1.imshow(np.log10(norm_dens[id][:,:,frame]),origin = "lower",cmap="inferno", extent = [0,35,0,35], vmin = d_vmin, vmax = d_vmax)
    
    # Swap out particle slice
    z_min, z_max = frame * dl, (frame + 1) * dl
    mask = (coords_z0[:,2] > z_min) & (coords_z0[:,2] < z_max)
    x, y = coords_z0[:, 0][mask], coords_z0[:, 1][mask]
    
    new_counts, _, _ = np.histogram2d(x, y, bins=256, range=[[0,35], [0,35]])
    ax2.imshow(new_counts.T, origin="lower", cmap="inferno", norm=LogNorm(vmin=1, vmax=10000), extent=[0,35,0,35])

    for ax in (ax1, ax2):
        ax.set_xlabel("x [cMpc/h]")
        ax.set_ylabel("y [cMpc/h]")
        ax.set_aspect('equal')
        ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
        ax.xaxis.set_major_locator(ticker.MultipleLocator(5))

# Set frames=128 to go through all your data slices
anim = animation.FuncAnimation(fig, update, frames=128, interval=50)

anim.save('/Users/users/roana/roana/BSc_Thesis/Animations/psdtef_nbody.mp4', writer='ffmpeg', fps=10)
print("2D Animation successfully saved!")

In [ ]:
from matplotlib.lines import Line2D

id = 0

coords_z0 = il.snapshot.loadSubset(basePath, snap_list[id], 'dm', fields = ['Coordinates'])/1000 

fig = plt.figure()
ax1 = fig.add_subplot(111)

# --- 1. DRAW PLOTS, LABELS, AND COLORBARS ONCE ---
d_vmin = np.min(np.log10(norm_dens))
d_vmax = np.max(np.log10(norm_dens))
im1 = ax1.imshow(np.log10(norm_dens[id][:,:,0]), origin="lower", cmap="inferno", extent=[0,35,0,35], vmin = d_vmin, vmax = d_vmax)
fig.colorbar(im1, ax=ax1, label=r"$\log (1+\delta)$")


legend_elements = [
    Line2D([0], [0], color=colors[0], lw=2, label='Nodes'),
    Line2D([0], [0], color=colors[1], lw=2, label='Filaments'),
    Line2D([0], [0], color=colors[2], lw=2, label='Walls')
]
ax1.legend(handles=legend_elements, loc="upper right", 
          facecolor="black", labelcolor="white", framealpha=0.8, edgecolor="none")

# Set labels once so we don't have to write them 128 times

ax1.set_xlabel("x [cMpc/h]")
ax1.set_ylabel("y [cMpc/h]")
ax1.set_aspect('equal')
ax1.yaxis.set_major_locator(ticker.MultipleLocator(5))
ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))

plt.tight_layout()

# --- 2. JUST LOOP THROUGH THE 128 SLICES ---
def update(frame):

    slice_index = frame
    # Swap out density slice
    ax1.clear()

    im1 = ax1.imshow(np.log10(norm_dens[id][:,:,slice_index]),origin = "lower",cmap="inferno", extent = [0,35,0,35], vmin = d_vmin, vmax = d_vmax)
    ax1.set_xlabel("x [cMpc/h]")
    ax1.set_ylabel("y [cMpc/h]")
    ax1.set_aspect('equal')
    
    ax1.contour(walls[id][:,:,frame], colors = colors[2], extent = [0,35,0,35], linewidths=.31, antialiased=True)
    ax1.contour(filaments[id][:,:,frame], colors = colors[1], extent = [0,35,0,35], linewidths=.31, antialiased=True)
    ax1.contour(nodes[id][:,:,frame], colors = colors[0], extent = [0,35,0,35], linewidths=.31, antialiased=True)
    ax1.legend(handles=legend_elements, loc="upper right", 
          facecolor="black", labelcolor="white", framealpha=0.8, edgecolor="none")

# Set frames=128 to go through all your data slices
anim = animation.FuncAnimation(fig, update, frames=128, interval=50)

anim.save('/Users/users/roana/roana/BSc_Thesis/Animations/nexus_slices.mp4', writer='ffmpeg', fps=10)
print("2D Animation successfully saved!")

In [ ]:
from matplotlib.lines import Line2D

id = 0

coords_z0 = il.snapshot.loadSubset(basePath, snap_list[id], 'dm', fields = ['Coordinates'])/1000 

fig = plt.figure()
ax1 = fig.add_subplot(111)

# --- 1. DRAW PLOTS, LABELS, AND COLORBARS ONCE ---
d_vmin = np.min(np.log10(norm_dens))
d_vmax = np.max(np.log10(norm_dens))
im1 = ax1.imshow(np.log10(norm_dens[id][:,:,0]), origin="lower", cmap="inferno", extent=[0,35,0,35], vmin = d_vmin, vmax = d_vmax)
fig.colorbar(im1, ax=ax1, label=r"$\log (1+\delta)$")


plt.tight_layout()

def update(frame):

    # Swap out density slice
    ax1.clear()

    ax1.imshow(np.log10(norm_dens[17-frame][:,:,78]),origin = "lower",cmap="inferno", extent = [0,35,0,35], vmin = d_vmin, vmax = d_vmax)
    ax1.set_xlabel("x [cMpc/h]")
    ax1.set_ylabel("y [cMpc/h]")
    ax1.set_aspect('equal')
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    

# Set frames=128 to go through all your data slices
anim = animation.FuncAnimation(fig, update, frames=18, interval=50)

anim.save('/Users/users/roana/roana/BSc_Thesis/Animations/d_fld_evol.mp4', writer='ffmpeg', fps=10)
print("2D Animation successfully saved!")

In [ ]:
from matplotlib.lines import Line2D

fig = plt.figure()
ax1 = fig.add_subplot(111)


plt.tight_layout()

def update(frame):

    # Swap out density slice
    ax1.clear()

    total = nodes[17-frame]*4 + filaments[17-frame]*3 + walls[17-frame]*2 + voids[17-frame]
    ax1.imshow(total[:,:,78], origin="lower", cmap="viridis_r", vmax = 4, vmin = 1, extent=[0,35,0,35])

    ax1.set_xlabel("x [cMpc/h]")
    ax1.set_ylabel("y [cMpc/h]")
    ax1.set_aspect('equal')
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(5))
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    

# Set frames=128 to go through all your data slices
anim = animation.FuncAnimation(fig, update, frames=18, interval=50)

anim.save('/Users/users/roana/roana/BSc_Thesis/Animations/nexus_evol.mp4', writer='ffmpeg', fps=10)
print("2D Animation successfully saved!")